# Pytorch Dataset & Dataloader demo

In [1]:
from sklearn.datasets import make_classification
X, y = make_classification(n_samples=10, n_features=2, n_informative=2, n_redundant=0, n_classes=2)

In [2]:
X

array([[-1.3106333 ,  1.22521864],
       [-1.43730299, -2.38591715],
       [-2.38778309,  1.17697607],
       [ 0.97490433,  0.93064339],
       [-0.35351474, -0.40288151],
       [-0.27966995, -0.16320641],
       [-1.80059795, -1.37691604],
       [ 1.97181908, -3.18783207],
       [ 2.13395771, -4.05722884],
       [ 2.01330711, -3.70176335]])

In [3]:
y

array([0, 0, 0, 1, 0, 1, 0, 1, 1, 1])

In [4]:
# Convert to pytorch tensors
import torch
X = torch.tensor(X, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.long)

In [17]:
from torch.utils.data import Dataset, DataLoader
class CustomDataset(Dataset):
  def __init__(self, features, labels):
    self.features = features
    self.labels = labels

  def __len__(self):
    return self.features.shape[0]

  def __getitem__(self, index):
    return self.features[index], self.labels[index]

In [18]:
dataset = CustomDataset(X, y)

In [19]:
len(dataset)

10

In [20]:
dataset[1]

(tensor([-1.4373, -2.3859]), tensor(0))

In [22]:
dataloader = DataLoader(dataset, batch_size=2, shuffle=True)
for batch_features, batch_labels in dataloader:
  print(batch_features)
  print(batch_labels)

tensor([[-1.4373, -2.3859],
        [ 2.0133, -3.7018]])
tensor([0, 1])
tensor([[ 0.9749,  0.9306],
        [-1.3106,  1.2252]])
tensor([1, 0])
tensor([[-0.2797, -0.1632],
        [-0.3535, -0.4029]])
tensor([1, 0])
tensor([[-2.3878,  1.1770],
        [ 1.9718, -3.1878]])
tensor([0, 1])
tensor([[ 2.1340, -4.0572],
        [-1.8006, -1.3769]])
tensor([1, 0])


# Dataset & Dataloader in Breast Cancer dataset

In [23]:
import pandas as pd
import numpy as np
df = pd.read_csv('https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv')
df.head(3)

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.8,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.6,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.9,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.8,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.0,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.5,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN


In [24]:
df = df.drop(columns=['id', 'Unnamed: 32'])

In [25]:
from sklearn.model_selection import train_test_split
X = df.drop(columns=['diagnosis'])
y = df['diagnosis']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [26]:
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

In [27]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)

In [28]:
# Convert numpy arrays to PyTorch tensors
import torch
X_train = torch.from_numpy(X_train).to(torch.float32)
X_test = torch.from_numpy(X_test).to(torch.float32)
y_train = torch.from_numpy(y_train).to(torch.float32)
y_test = torch.from_numpy(y_test).to(torch.float32)

In [29]:
from torch.utils.data import Dataset, DataLoader
class CustomDataset(Dataset):
  def __init__(self, features, labels):
    self.features = features
    self.labels = labels

  def __len__(self):
    return self.features.shape[0]

  def __getitem__(self, index):
    return self.features[index], self.labels[index]

In [31]:
train_ds = CustomDataset(X_train, y_train)
test_ds = CustomDataset(X_test, y_test)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=True)

In [32]:
import torch
import torch.nn as nn

class Neural_Network(nn.Module):
  def __init__(self, num_features):
    super().__init__()
    self.network = nn.Sequential(
        nn.Linear(num_features, 1),
        nn.Sigmoid()
    )

  def forward(self, num_features):
    out = self.network(num_features)
    return out

In [33]:
learning_rate = 0.1
epochs = 25

In [34]:
model = Neural_Network(X_train.shape[1])
loss_function = nn.BCELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

for epoch in range(epochs):
  for batch_features, batch_labels in train_loader:
    y_pred = model(batch_features)
    loss = loss_function(y_pred, batch_labels.reshape(-1, 1))
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

  print(f'Epoch: {epoch+1}, Loss:{loss.item()}')

Epoch: 1, Loss:0.15198291838169098
Epoch: 2, Loss:0.3287431299686432
Epoch: 3, Loss:0.08136874437332153
Epoch: 4, Loss:0.08613216876983643
Epoch: 5, Loss:0.0642290934920311
Epoch: 6, Loss:0.017597287893295288
Epoch: 7, Loss:0.06988651305437088
Epoch: 8, Loss:0.1473492532968521
Epoch: 9, Loss:0.027003096416592598
Epoch: 10, Loss:0.057874519377946854
Epoch: 11, Loss:0.10082081705331802
Epoch: 12, Loss:0.012964689172804356
Epoch: 13, Loss:0.07959357649087906
Epoch: 14, Loss:0.12432064116001129
Epoch: 15, Loss:0.03966430202126503
Epoch: 16, Loss:0.055152777582407
Epoch: 17, Loss:0.02156134508550167
Epoch: 18, Loss:0.08836402744054794
Epoch: 19, Loss:0.004609259776771069
Epoch: 20, Loss:0.013596327044069767
Epoch: 21, Loss:0.013089762069284916
Epoch: 22, Loss:0.03784114494919777
Epoch: 23, Loss:0.0039883931167423725
Epoch: 24, Loss:0.002507037715986371
Epoch: 25, Loss:0.009910181164741516


In [35]:
model.eval()
accuracy_list = []

with torch.no_grad():
  for batch_features, batch_labels in test_loader:
    y_pred = model(batch_features)
    y_pred = (y_pred > 0.9).float()
    accuracy = (y_pred == batch_labels.reshape(-1, 1)).float().mean().item()
    accuracy_list.append(accuracy)
accuracy = np.mean(accuracy_list)
print(f'Accuracy: {accuracy}')

Accuracy: 0.8923611044883728
